In [0]:
%pip install --upgrade youtube-transcript-api google-genai
dbutils.library.restartPython()

In [0]:
%pip install google-genai youtube-transcript-api google-api-python-client
dbutils.library.restartPython()

In [0]:
# pip uninstall youtube-transcript-api -y
%pip install youtube-transcript-api==0.6.2


In [0]:
dbutils.library.restartPython()

In [0]:
from youtube_transcript_api import YouTubeTranscriptApi

print("Fetching transcript...")
transcript = YouTubeTranscriptApi.get_transcript("DInMru2Eq6E") # Jenny's 1st video ID
print(transcript[:500]) # First 500 characters print chestundi
print("\n✅ SUCCESS! Transcript is working locally.")

In [0]:
import os
import json
import time
from google import genai
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi

# ==========================================
# 1. SETUP YOUR API KEYS HERE
# ==========================================
GEMINI_API_KEY = "MEE_GEMINI_API_KEY_IKKADA_PETTU"
YOUTUBE_API_KEY = "MEE_YOUTUBE_API_KEY_IKKADA_PETTU"

PLAYLIST_ID = "PLdo5W4Nhv31bZSiqiOL5ta39vSnBxpOPT" # Jenny's Python
LOCAL_DB_FILE = "course_local_db.json"
FINAL_OUTPUT_FILE = "py_notes_final.json"

client = genai.Client(api_key=GEMINI_API_KEY)
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

# ==========================================
# 2. FUNCTIONS
# ==========================================
def get_playlist_videos():
    print("Fetching Playlist from YouTube...")
    videos = []
    next_token = None
    topic_index = 0
    while True:
        req = youtube.playlistItems().list(part="snippet", playlistId=PLAYLIST_ID, maxResults=50, pageToken=next_token)
        res = req.execute()
        for item in res['items']:
            title = item['snippet']['title']
            if title not in ["Private video", "Deleted video"]:
                videos.append({
                    "topic_id": f"topic_{topic_index}",
                    "video_id": item['snippet']['resourceId']['videoId'],
                    "title": title,
                    "is_processed": False,
                    "ai_data": None
                })
                topic_index += 1
        next_token = res.get('nextPageToken')
        if not next_token: break
    return videos

# 🔥 BULLET-PROOF TRANSCRIPT EXTRACTOR 🔥
def get_transcript(video_id):
    try:
        # Attempt 1: Fetch default English transcript
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
        return " ".join([item['text'] for item in transcript_list])
    except Exception as e1:
        try:
            # Attempt 2: If default fails, fetch ANY available transcript (Auto-generated/Hindi/etc)
            transcripts = YouTubeTranscriptApi.list_transcripts(video_id)
            for t in transcripts:
                # Grab the very first transcript available and return it
                return " ".join([item['text'] for item in t.fetch()])
        except Exception as e2:
            print(f"      ⚠️ Reason 1: {e1}")
            print(f"      ⚠️ Reason 2: {e2}")
            return None
        
def generate_ai_content(transcript, title):
    prompt = f"""
    You are an Elite Silicon Valley Coding Instructor and a Master UX Content Designer for a premium educational app.
    Your objective is to transform the raw video transcript into a world-class, highly engaging, 'Topper-Level' study guide and adaptive practice test.
    
    Topic Context: '{title}'
    
    OUTPUT FORMAT: 
    Return EXACTLY a valid JSON object. Do not include ```json blocks or any plain text outside the JSON.
    
    JSON STRUCTURE & STRICT REQUIREMENTS:
    
    1. "suggested_name" (String): 
       - A clean, concise, professional title. 
       - Remove all YouTube fluff, channel names, pipe characters (|), and lecture numbers. 
       - Example: "History of Python | Python Tutorials for Beginners #lec2" -> "History of Python".
       
    2. "markdown_notes" (String):
       - Create visually stunning, easy-to-scan, and highly educational markdown text.
       - Structure it perfectly using this flow:
         * 🎯 **Overview:** A crisp 2-line summary of what this topic covers.
         * 📌 **Core Concepts:** Use clear sub-headings (###) and bullet points. Bold important keywords.
         * 💻 **Code in Action:** Provide 1-2 highly practical Python code examples. You MUST include detailed, easy-to-understand inline comments (`#`) explaining the logic step-by-step.
         * 💡 **Pro-Tip / ⚠️ Common Pitfall:** Add a small practical tip or a warning about a common mistake students make.
       - EXERCISE RULE: If the transcript is a "Coding Exercise", ignore the standard flow and structure the notes as: "Problem Statement" -> "Logic & Approach" -> "Solution Code" -> "Code Walkthrough".
       
    3. "practice_test" (Array of Objects):
       - Provide EXACTLY 3 multiple-choice questions.
       - ADAPTIVE RULE: 
         - If theory-based: Ask conceptual questions to test understanding.
         - If code/exercise-based: Ask logic-based questions or "Predict the Output" questions (using small code snippets in the question text).
       - Object Format: {{"q": "Question text?", "opts": ["Option 1", "Option 2", "Option 3", "Option 4"], "ans": 0, "exp": "One-line clear explanation why this is correct."}}
       
    Transcript to process: 
    {transcript[:8000]}
    """
    try:
        response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt, config={'temperature': 0.2})
        res_text = response.text.replace('```json\n', '').replace('```', '').strip()
        return json.loads(res_text)
    except Exception as e:
        print(f"Gemini Error: {e}")
        return None
# ==========================================
# 3. THE MAIN ENGINE
# ==========================================
def run_local_engine():
    if os.path.exists(LOCAL_DB_FILE):
        with open(LOCAL_DB_FILE, 'r') as f:
            db = json.load(f)
        print("Loaded existing database from local file.")
    else:
        db = get_playlist_videos()
        with open(LOCAL_DB_FILE, 'w') as f:
            json.dump(db, f, indent=4)
            
    for video in db:
        if video['is_processed']:
            continue
            
        print(f"\n⏳ Processing: {video['title']}")
        
        transcript = get_transcript(video['video_id'])
        if not transcript:
            print(f"⏭️ Skipping (No Subtitles).")
            # Marking as true so it doesn't get stuck in a loop later
            video['is_processed'] = True 
            video['ai_data'] = {"suggested_name": video['title'], "markdown_notes": "Notes unavailable.", "practice_test": []}
        else:
            ai_data = generate_ai_content(transcript, video['title'])
            if ai_data:
                video['ai_data'] = ai_data
                video['is_processed'] = True
                print(f"✅ Success! Clean Name: {ai_data.get('suggested_name')}")
            else:
                print(f"❌ Gemini generation failed. Retrying later.")
                continue 
                
        # Save progress
        with open(LOCAL_DB_FILE, 'w') as f:
            json.dump(db, f, indent=4)
            
        time.sleep(4) 
        
    print("\n🎉 All Videos Processed! Exporting to Vercel format...")
    export_json = {"course_notes": {"py_notes_shared": {}}}
    
    for v in db:
        if v.get('ai_data'):
            export_json["course_notes"]["py_notes_shared"][v['topic_id']] = v['ai_data']
            
    with open(FINAL_OUTPUT_FILE, 'w') as f:
        json.dump(export_json, f, separators=(',', ':')) 
        
    print(f"🚀 DONE! You can now upload '{FINAL_OUTPUT_FILE}' to Vercel.")

if __name__ == "__main__":
    run_local_engine()

In [0]:
import json
import time
from google import genai
from youtube_transcript_api import YouTubeTranscriptApi

# 1. API Setup
GEMINI_API_KEY = dbutils.secrets.get(scope="courseify", key="gemini-api")
client = genai.Client(api_key=GEMINI_API_KEY)

# 2. Robust Transcript Function (🔥 UPDATED TO LATEST 2026 SYNTAX 🔥)
def get_transcript(video_id):
    try:
        # Patha 'get_transcript' badulu kotha 'fetch' method vadutunnam
        api = YouTubeTranscriptApi()
        transcript_list = api.fetch(video_id, languages=['en', 'en-IN', 'hi'])
        return " ".join([item['text'] for item in transcript_list])
    except Exception as e:
        print(f"Transcript Error for {video_id}: {e}")
        return None

# 3. Gemini Content Generator (THE ULTIMATE SUPER-PROMPT)
def generate_ai_content(transcript, title):
    prompt = f"""
    You are an Elite Silicon Valley Coding Instructor and a Master UX Content Designer for a premium educational app.
    Your objective is to transform the raw video transcript into a world-class, highly engaging, 'Topper-Level' study guide and adaptive practice test.
    
    Topic Context: '{title}'
    
    OUTPUT FORMAT: 
    Return EXACTLY a valid JSON object. Do not include ```json blocks or any plain text outside the JSON.
    
    JSON STRUCTURE & STRICT REQUIREMENTS:
    
    1. "suggested_name" (String): 
       - A clean, concise, professional title. 
       - Remove all YouTube fluff, channel names, pipe characters (|), and lecture numbers. 
       - Example: "History of Python | Python Tutorials for Beginners #lec2" -> "History of Python".
       
    2. "markdown_notes" (String):
       - Create visually stunning, easy-to-scan, and highly educational markdown text.
       - Structure it perfectly using this flow:
         * 🎯 **Overview:** A crisp 2-line summary of what this topic covers.
         * 📌 **Core Concepts:** Use clear sub-headings (###) and bullet points. Bold important keywords.
         * 💻 **Code in Action:** Provide 1-2 highly practical Python code examples. You MUST include detailed, easy-to-understand inline comments (`#`) explaining the logic step-by-step.
         * 💡 **Pro-Tip / ⚠️ Common Pitfall:** Add a small practical tip or a warning about a common mistake students make.
       - EXERCISE RULE: If the transcript is a "Coding Exercise", ignore the standard flow and structure the notes as: "Problem Statement" -> "Logic & Approach" -> "Solution Code" -> "Code Walkthrough".
       
    3. "practice_test" (Array of Objects):
       - Provide EXACTLY 3 multiple-choice questions.
       - ADAPTIVE RULE: 
         - If theory-based: Ask conceptual questions to test understanding.
         - If code/exercise-based: Ask logic-based questions or "Predict the Output" questions (using small code snippets in the question text).
       - Object Format: {{"q": "Question text?", "opts": ["Option 1", "Option 2", "Option 3", "Option 4"], "ans": 0, "exp": "One-line clear explanation why this is correct."}}
       
    Transcript to process: 
    {transcript[:8000]}
    """
    
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config={'temperature': 0.2}
        )
        res_text = response.text.replace('```json\n', '').replace('```', '').strip()
        return json.loads(res_text)
    except Exception as e:
        print(f"Gemini generation failed: {e}")
        return None

# 4. Main Processing Loop (BATCH MODE)
def process_pending_videos():
    # Only pick videos that haven't been processed AND haven't failed transcript extraction
    query = """
        SELECT video_links, topic 
        FROM courseify.default.courseify_data 
        WHERE notes = false AND (description IS NULL OR description != 'TRANSCRIPT_FAILED')
        LIMIT 5
    """
    pending_videos = spark.sql(query).collect()
    
    if not pending_videos:
        print("✅ All videos are fully processed or checked!")
        return
        
    print(f"Processing a batch of {len(pending_videos)} videos...\n")
    
    for row in pending_videos:
        v_id = row['video_links']
        v_title = row['topic']
        print(f"⏳ Processing: {v_title}")
        
        # Step A: Get Transcript (Using new Fetch logic)
        transcript = get_transcript(v_id)
        if not transcript:
            print(f"⏭️ Skipping (Transcript Unavailable). Marking as Failed in DB.")
            spark.sql(f"UPDATE courseify.default.courseify_data SET description = 'TRANSCRIPT_FAILED' WHERE video_links = '{v_id}'")
            continue
            
        # Step B: Generate Notes, QA & Clean Name
        ai_data = generate_ai_content(transcript, v_title)
        
        if ai_data:
            escaped_notes = ai_data.get('markdown_notes', '').replace("'", "\\'")
            escaped_qa = json.dumps(ai_data.get('practice_test', [])).replace("'", "\\'")
            escaped_desc = transcript[:1000].replace("'", "\\'") 
            
            clean_name = ai_data.get('suggested_name', v_title)
            escaped_name = clean_name.replace("'", "\\'")
            
            # Step C: Update the database
            update_query = f"""
                UPDATE courseify.default.courseify_data 
                SET description = '{escaped_desc}', 
                    notes_data = '{escaped_notes}', 
                    qa_data = '{escaped_qa}', 
                    suggested_name = '{escaped_name}',
                    notes = true, 
                    qa = true
                WHERE video_links = '{v_id}'
            """
            spark.sql(update_query)
            print(f"✅ Success! Cleaned Name: '{clean_name}'")
        else:
            print(f"❌ Failed AI Generation for: {v_title}")
            
        # SLEEP to respect API limits
        time.sleep(5) 

    print("\nBatch Complete! Run this cell again to process the next 5 videos.")

# Run the engine
process_pending_videos()

In [0]:
import json
import time
from google import genai
from youtube_transcript_api import YouTubeTranscriptApi

# 1. API Setup
GEMINI_API_KEY = dbutils.secrets.get(scope="courseify", key="gemini-api")
client = genai.Client(api_key=GEMINI_API_KEY)

# 2. Robust Transcript Function
def get_transcript(video_id):
    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=['en', 'en-IN', 'hi'])
        return " ".join([item['text'] for item in transcript_list])
    except Exception as e:
        print(f"Transcript Error for {video_id}: {e}")
        return None

# 3. Gemini Content Generator (THE ULTIMATE SUPER-PROMPT)
def generate_ai_content(transcript, title):
    prompt = f"""
    You are an Elite Silicon Valley Coding Instructor and a Master UX Content Designer for a premium educational app.
    Your objective is to transform the raw video transcript into a world-class, highly engaging, 'Topper-Level' study guide and adaptive practice test.
    
    Topic Context: '{title}'
    
    OUTPUT FORMAT: 
    Return EXACTLY a valid JSON object. Do not include ```json blocks or any plain text outside the JSON.
    
    JSON STRUCTURE & STRICT REQUIREMENTS:
    
    1. "suggested_name" (String): 
       - A clean, concise, professional title. 
       - Remove all YouTube fluff, channel names, pipe characters (|), and lecture numbers. 
       - Example: "History of Python | Python Tutorials for Beginners #lec2" -> "History of Python".
       
    2. "markdown_notes" (String):
       - Create visually stunning, easy-to-scan, and highly educational markdown text.
       - Structure it perfectly using this flow:
         * 🎯 **Overview:** A crisp 2-line summary of what this topic covers.
         * 📌 **Core Concepts:** Use clear sub-headings (###) and bullet points. Bold important keywords.
         * 💻 **Code in Action:** Provide 1-2 highly practical Python code examples. You MUST include detailed, easy-to-understand inline comments (`#`) explaining the logic step-by-step.
         * 💡 **Pro-Tip / ⚠️ Common Pitfall:** Add a small practical tip or a warning about a common mistake students make.
       - EXERCISE RULE: If the transcript is a "Coding Exercise", ignore the standard flow and structure the notes as: "Problem Statement" -> "Logic & Approach" -> "Solution Code" -> "Code Walkthrough".
       
    3. "practice_test" (Array of Objects):
       - Provide EXACTLY 3 multiple-choice questions.
       - ADAPTIVE RULE: 
         - If theory-based: Ask conceptual questions to test understanding.
         - If code/exercise-based: Ask logic-based questions or "Predict the Output" questions (using small code snippets in the question text).
       - Object Format: {{"q": "Question text?", "opts": ["Option 1", "Option 2", "Option 3", "Option 4"], "ans": 0, "exp": "One-line clear explanation why this is correct."}}
       
    Transcript to process: 
    {transcript[:8000]}
    """
    
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config={'temperature': 0.2}
        )
        res_text = response.text.replace('```json\n', '').replace('```', '').strip()
        return json.loads(res_text)
    except Exception as e:
        print(f"Gemini generation failed: {e}")
        return None

# 4. Main Processing Loop (BATCH MODE)
def process_pending_videos():
    # FIX: Fetching 'topic' (Video Title). Also ignoring previously failed transcripts so we don't get stuck.
    query = """
        SELECT video_links, topic 
        FROM courseify.default.courseify_data 
        WHERE notes = false AND (description IS NULL OR description != 'TRANSCRIPT_FAILED')
        LIMIT 5
    """
    pending_videos = spark.sql(query).collect()
    
    if not pending_videos:
        print("✅ All videos are fully processed or checked!")
        return
        
    print(f"Processing a batch of {len(pending_videos)} videos...\n")
    
    for row in pending_videos:
        v_id = row['video_links']
        v_title = row['topic'] # FIX: Actual Video Title
        print(f"⏳ Processing: {v_title}")
        
        # Step A: Get Transcript
        transcript = get_transcript(v_id)
        if not transcript:
            print(f"⏭️ Skipping (Transcript Unavailable). Marking as Failed in DB.")
            spark.sql(f"UPDATE courseify.default.courseify_data SET description = 'TRANSCRIPT_FAILED' WHERE video_links = '{v_id}'")
            continue
            
        # Step B: Generate Notes, QA & Clean Name
        ai_data = generate_ai_content(transcript, v_title)
        
        if ai_data:
            escaped_notes = ai_data.get('markdown_notes', '').replace("'", "\\'")
            escaped_qa = json.dumps(ai_data.get('practice_test', [])).replace("'", "\\'")
            escaped_desc = transcript[:1000].replace("'", "\\'") 
            
            clean_name = ai_data.get('suggested_name', v_title)
            escaped_name = clean_name.replace("'", "\\'")
            
            # Step C: Update the database (Only on true success)
            update_query = f"""
                UPDATE courseify.default.courseify_data 
                SET description = '{escaped_desc}', 
                    notes_data = '{escaped_notes}', 
                    qa_data = '{escaped_qa}', 
                    suggested_name = '{escaped_name}',
                    notes = true, 
                    qa = true
                WHERE video_links = '{v_id}'
            """
            spark.sql(update_query)
            print(f"✅ Success! Cleaned Name: '{clean_name}'")
        else:
            print(f"❌ Failed AI Generation for: {v_title}")
            
        # SLEEP to respect API limits
        time.sleep(5) 

    print("\nBatch Complete! Run this cell again to process the next 5 videos.")

# Run the engine
process_pending_videos()

In [0]:
# import json
# import time
# from google import genai
# from youtube_transcript_api import YouTubeTranscriptApi

# # 1. API Setup (Using the new SDK)
# GEMINI_API_KEY = dbutils.secrets.get(scope="courseify", key="gemini-api")
# client = genai.Client(api_key=GEMINI_API_KEY)

# # 2. Get Transcript Function
# def get_transcript(video_id):
#     try:
#         # YouTube auto-generates subtitles, fetching them here
#         transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=['en', 'en-IN', 'hi'])
#         return " ".join([item['text'] for item in transcript_list])
#     except Exception as e:
#         print(f"Transcript failed for {video_id}: {e}")
#         return None

# # # 3. Gemini Content Generator
# # def generate_ai_content(transcript, title):
# #     prompt = f"""
# #     You are an expert coding tutor. Based ONLY on the following video transcript for the topic '{title}', generate study notes, a practice test, and a clean suggested topic name.
    
# #     REQUIREMENTS:
# #     - "suggested_name": A clean, concise, professional title for this topic. Remove all YouTube-specific fluff, channel names, pipe characters (|), and lecture numbers (like #lec1). Example: "History of Python | Python Tutorials for Beginners #lec2" should become just "History of Python".
# #     - "markdown_notes": Beautiful markdown text (headings, bullet points, 1-2 Python code examples).
# #     - "practice_test": Array of exactly 2 multiple choice questions based on the transcript.
    
# #     Return EXACTLY a valid JSON object with keys: "suggested_name", "markdown_notes", and "practice_test".
# #     Format for practice test objects: {{"q": "...", "opts": ["A", "B", "C", "D"], "ans": 0, "exp": "..."}}
    
# #     Transcript: {transcript[:8000]}
# #     """
    
# #     try:
# #         response = client.models.generate_content(
# #             model='gemini-2.5-flash',
# #             contents=prompt
# #         )
# #         # Clean the response to parse JSON safely
# #         res_text = response.text.replace('```json\n', '').replace('```', '').strip()
# #         return json.loads(res_text)
# #     except Exception as e:
# #         print(f"Gemini generation failed: {e}")
# #         return None

# # 3. Gemini Content Generator (THE ULTIMATE SUPER-PROMPT)
# def generate_ai_content(transcript, title):
#     prompt = f"""
#     You are an Elite Silicon Valley Coding Instructor and a Master UX Content Designer for a premium educational app.
#     Your objective is to transform the raw video transcript into a world-class, highly engaging, 'Topper-Level' study guide and adaptive practice test.
    
#     Topic Context: '{title}'
    
#     OUTPUT FORMAT: 
#     Return EXACTLY a valid JSON object. Do not include ```json blocks or any plain text outside the JSON.
    
#     JSON STRUCTURE & STRICT REQUIREMENTS:
    
#     1. "suggested_name" (String): 
#        - A clean, concise, professional title. 
#        - Remove all YouTube fluff, channel names, pipe characters (|), and lecture numbers. 
#        - Example: "History of Python | Python Tutorials for Beginners #lec2" -> "History of Python".
       
#     2. "markdown_notes" (String):
#        - Create visually stunning, easy-to-scan, and highly educational markdown text.
#        - Structure it perfectly using this flow:
#          * 🎯 **Overview:** A crisp 2-line summary of what this topic covers.
#          * 📌 **Core Concepts:** Use clear sub-headings (###) and bullet points. Bold important keywords.
#          * 💻 **Code in Action:** Provide 1-2 highly practical Python code examples. You MUST include detailed, easy-to-understand inline comments (`#`) explaining the logic step-by-step.
#          * 💡 **Pro-Tip / ⚠️ Common Pitfall:** Add a small practical tip or a warning about a common mistake students make.
#        - EXERCISE RULE: If the transcript is a "Coding Exercise", ignore the standard flow and structure the notes as: "Problem Statement" -> "Logic & Approach" -> "Solution Code" -> "Code Walkthrough".
       
#     3. "practice_test" (Array of Objects):
#        - Provide EXACTLY more than 5 multiple-choice questions.
#        - ADAPTIVE RULE: 
#          - If theory-based: Ask conceptual questions to test understanding.
#          - If code/exercise-based: Ask logic-based questions or "Predict the Output" questions (using small code snippets in the question text).
#        - Object Format: {{"q": "Question text?", "opts": ["Option 1", "Option 2", "Option 3", "Option 4"], "ans": 0, "exp": "One-line clear explanation why this is correct."}}
       
#     Transcript to process: 
#     {transcript[:8000]}
#     """
    
#     try:
#         response = client.models.generate_content(
#             model='gemini-2.5-flash',
#             contents=prompt,
#             config={'temperature': 0.2} # Highly focused and deterministic output
#         )
#         res_text = response.text.replace('```json\n', '').replace('```', '').strip()
#         return json.loads(res_text)
#     except Exception as e:
#         print(f"Gemini generation failed: {e}")
#         return None

# # 4. Main Processing Loop (BATCH MODE: Process 5 videos at a time)
# def process_pending_videos():
#     # Fetch only 5 videos where notes are not yet generated
#     pending_videos = spark.sql("SELECT video_links, title FROM courseify.default.courseify_data WHERE notes = false LIMIT 5").collect()
    
#     if not pending_videos:
#         print("✅ All videos are fully processed with Notes, QA, and Clean Names!")
#         return
        
#     print(f"Processing a batch of {len(pending_videos)} videos...\n")
    
#     for row in pending_videos:
#         v_id = row['video_links']
#         v_title = row['title']
#         print(f"⏳ Processing: {v_title}")
        
#         # Step A: Get Transcript
#         transcript = get_transcript(v_id)
#         if not transcript:
#             # If no subtitles exist, mark as processed but empty
#             spark.sql(f"UPDATE courseify.default.courseify_data SET notes = true, qa = true, notes_data = 'Notes not available', qa_data = '[]', suggested_name = '{v_title}' WHERE video_links = '{v_id}'")
#             continue
            
#         # Step B: Generate Notes, QA & Clean Name
#         ai_data = generate_ai_content(transcript, v_title)
        
#         if ai_data:
#             # Escape single quotes for SQL insertion
#             escaped_notes = ai_data['markdown_notes'].replace("'", "\\'")
#             escaped_qa = json.dumps(ai_data['practice_test']).replace("'", "\\'")
#             escaped_desc = transcript[:1000].replace("'", "\\'") # Save a snippet of transcript
            
#             # Use suggested name, or fallback to original title if missing
#             clean_name = ai_data.get('suggested_name', v_title)
#             escaped_name = clean_name.replace("'", "\\'")
            
#             # Step C: Update the database
#             update_query = f"""
#                 UPDATE courseify.default.courseify_data 
#                 SET description = '{escaped_desc}', 
#                     notes_data = '{escaped_notes}', 
#                     qa_data = '{escaped_qa}', 
#                     suggested_name = '{escaped_name}',
#                     notes = true, 
#                     qa = true
#                 WHERE video_links = '{v_id}'
#             """
#             spark.sql(update_query)
#             print(f"✅ Success! Cleaned Name: '{clean_name}'")
#         else:
#             print(f"❌ Failed AI Generation for: {v_title}")
            
#         # 🚨 SLEEP for 5 seconds to avoid Gemini Free Tier API Rate Limits
#         time.sleep(5) 

#     print("\nBatch Complete! Run this cell again to process the next 5 videos.")

# # Run the engine
# process_pending_videos()